# z302 — Etapa 2: Feature Engineering

Input  : `z301_preprocessed.parquet`
Output : `z302_features.parquet` + `z302_inferencia.parquet`

Responsabilidades:
- Lags consecutivos de `tn` (lag_0 = mes actual, lag_1 … lag_N)
- Escalado rolling (media, std) calculado hacia atrás (sin leakage)
- Market share usando la `cat2` que ya viene de z301
- Encoding de variables categóricas (cat1, cat2, cat3, brand) para LGBM
- Target = tn en t+horizonte
- Split train / inferencia

## 0. Ambiente

In [ ]:
import os, shutil, subprocess

# El bucket ya está montado por gcsfuse en /home/ds/buckets/b1.
# SQLite de Optuna va al disco LOCAL (/home/ds), no al bucket (gcsfuse no soporta locks).
BASE       = '/home/ds/buckets/b1'
LOCAL_HOME = '/home/ds'

os.makedirs(f'{BASE}/exp',      exist_ok=True)
os.makedirs(f'{BASE}/datasets', exist_ok=True)

# Kaggle auth: usar el de ~/.kaggle si ya existe; si no, buscarlo en el bucket
kaggle_dst = os.path.expanduser('~/.kaggle/kaggle.json')
os.makedirs(os.path.dirname(kaggle_dst), exist_ok=True)
if os.path.exists(kaggle_dst):
    os.chmod(kaggle_dst, 0o600)
    print('Kaggle auth OK (ya estaba en ~/.kaggle)')
else:
    _encontrado = False
    for cand in [f'{BASE}/kaggle.json', f'{BASE}/kaggle/kaggle.json']:
        if os.path.exists(cand):
            shutil.copy(cand, kaggle_dst)
            os.chmod(kaggle_dst, 0o600)
            print(f'Kaggle auth OK (copiado de {cand})')
            _encontrado = True
            break
    if not _encontrado:
        print('⚠️  kaggle.json no encontrado. Subilo a ~/.kaggle/kaggle.json o al bucket.')

def descargar(archivo):
    url = f'https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/{archivo}'
    dst = f'{BASE}/datasets/{archivo}'
    if not os.path.exists(dst):
        subprocess.run(['wget', url, '-O', dst], check=True)
    print(f'✅ {archivo}')

descargar('sell-in.txt.gz')
descargar('tb_productos.txt')
descargar('tb_stocks.txt')
descargar('product_id_apredecir201912.txt')

In [ ]:
!pip install -q uv
!uv pip install -q pyarrow fastparquet

## 1. Parámetros — palancas

In [ ]:
import os

PARAM = {
    'experimento': 'z302',

    # ── PALANCA 1 (debe coincidir con z301) ───────────────────────
    'modo_agrupacion': 'producto',

    # ── PALANCA 3: lags consecutivos ───────────────────────────
    'lags': list(range(0, 12)),

    # ── PALANCA 4: ventana rolling ────────────────────────────
    'ventana_rolling': None,

    # ── PALANCA 5: tipo de target ─────────────────────────────
    'tipo_target': 'nivel',

    # ── PALANCA 6: market share ──────────────────────────────
    'incluir_market_share': True,

    # ── PALANCA 7: horizonte ─────────────────────────────────
    'horizonte': 2,

    'cols_categoricas': ['cat1', 'cat2', 'cat3', 'brand'],
}

# Paths derivados del modo: lee el z301 del modo correcto, escribe con el modo en el nombre
MODO = PARAM['modo_agrupacion']
PARAM['path_input']  = f'/home/ds/buckets/b1/exp/z301_preprocessed_{MODO}.parquet'
PARAM['path_output'] = f'/home/ds/buckets/b1/exp/z302_features_{MODO}.parquet'

ruta_exp = '/home/ds/buckets/b1/exp/' + PARAM['experimento']
os.makedirs(ruta_exp, exist_ok=True)
os.chdir(ruta_exp)
print('Parámetros:', PARAM)
print('Input:',  PARAM['path_input'])
print('Output:', PARAM['path_output'])

## 2. Carga

In [ ]:
import polars as pl
import numpy as np

df = pl.read_parquet(PARAM['path_input'])
print(f'Input: {df.shape}')
print(df.schema)
df.head(3)

## 3. Market share y categoría (vienen de z301)

Estas features se calculan en z301 sobre el universo completo. Acá solo se verifican.

In [ ]:
# El market share y tn_cat2_total ahora vienen calculados de z301
# (sobre el universo COMPLETO de productos). Acá solo verificamos.
for c in ['tn_cat2_total', 'market_share', 'productos_activos_cat2', 'productos_nuevos_cat2_3m']:
    estado = 'OK' if c in df.columns else 'FALTA (re-correr z301)'
    print(f'  {c}: {estado}')

## 4. Encoding de categóricas para LGBM

LGBM no acepta strings. Convertimos cat1/cat2/cat3/brand a códigos enteros con `.to_physical()`.
El mapeo es consistente porque se aplica sobre todo el dataset de una vez.

In [ ]:
for col in PARAM['cols_categoricas']:
    if col in df.columns:
        df = df.with_columns(
            pl.col(col).cast(pl.Categorical).to_physical().cast(pl.Int32).alias(col)
        )
        print(f'  {col}: encodeado')

# descripcion no sirve como feature
if 'descripcion' in df.columns:
    df = df.drop('descripcion')
    print('  descripcion: eliminada (texto libre)')

print('Encoding completo.')

## 5. Lags consecutivos de tn

`lag_0` = mes actual (= tn, sin desplazar). `lag_k` = tn de hace k meses.
Calculados dentro de cada `agrupa_id`.

In [ ]:
df = df.sort(['agrupa_id', 'periodo'])

lag_exprs = [
    pl.col('tn').shift(k).over('agrupa_id').alias(f'lag_{k}')
    for k in PARAM['lags']
]
df = df.with_columns(lag_exprs)

print('Lags calculados:', [f'lag_{k}' for k in PARAM['lags']])
df.filter(pl.col('agrupa_id') == df['agrupa_id'][0]).select(
    ['periodo', 'tn'] + [f'lag_{k}' for k in PARAM['lags'][:4]]
).head(6)

## 5b. Features de tendencia

Diferencias, pendientes, ratios y medias móviles. Le dan al modelo señal de **dirección** (si el producto sube o baja), atacando el sesgo de sobreestimación en productos en declive.

- `delta_1/2/3`: cambio respecto a 1, 2, 3 meses atrás
- `delta_anual`: cambio respecto a ~1 año (lag_11)
- `pendiente_3m`: tendencia de los últimos 3 meses
- `ratio_0_1`, `ratio_0_3`: crecimiento relativo
- `media_movil_3/6`: medias de 3 y 6 meses (sin leakage)
- `ms_delta_3`: cambio del market share vs 3 meses atrás

In [ ]:
df = df.sort(['agrupa_id', 'periodo'])

# Diferencias respecto a lags (señal de dirección)
df = df.with_columns([
    (pl.col('lag_0') - pl.col('lag_1')).alias('delta_1'),
    (pl.col('lag_0') - pl.col('lag_2')).alias('delta_2'),
    (pl.col('lag_0') - pl.col('lag_3')).alias('delta_3'),
    (pl.col('lag_0') - pl.col('lag_11')).alias('delta_anual'),
])

# Pendiente de los últimos 3 meses (aprox: (lag_0 - lag_2) / 2)
df = df.with_columns(
    ((pl.col('lag_0') - pl.col('lag_2')) / 2).alias('pendiente_3m')
)

# Ratios de crecimiento relativo (división protegida)
df = df.with_columns([
    pl.when(pl.col('lag_1') > 0).then(pl.col('lag_0') / pl.col('lag_1'))
      .otherwise(pl.lit(0.0)).alias('ratio_0_1'),
    pl.when(pl.col('lag_3') > 0).then(pl.col('lag_0') / pl.col('lag_3'))
      .otherwise(pl.lit(0.0)).alias('ratio_0_3'),
])

# Medias móviles de 3 y 6 meses (shift(1) para no incluir el período actual → sin leakage)
df = df.with_columns([
    pl.col('tn').shift(1).rolling_mean(window_size=3, min_periods=1)
      .over('agrupa_id').alias('media_movil_3'),
    pl.col('tn').shift(1).rolling_mean(window_size=6, min_periods=1)
      .over('agrupa_id').alias('media_movil_6'),
])

# Cambio del market share vs 3 meses atrás (captura canibalización indirecta)
if 'market_share' in df.columns:
    df = df.with_columns(
        (pl.col('market_share') - pl.col('market_share').shift(3).over('agrupa_id'))
        .alias('ms_delta_3')
    )

print('Features de tendencia calculadas.')
cols_nuevas = ['delta_1', 'delta_3', 'delta_anual', 'pendiente_3m',
               'ratio_0_1', 'media_movil_3', 'media_movil_6']
print(df.filter(pl.col('product_id') == 20001).select(['periodo', 'tn'] + cols_nuevas).head(6))

## 5c. Features de comportamiento e intermitencia

Capturan patrones de actividad y estabilidad de la serie (calculadas sobre `agrupa_id`, funcionan en ambos modos):
- `meses_activo`: meses desde la primera venta real
- `racha_ceros`: meses consecutivos en cero hasta el actual (intermitencia)
- `compro_igual_mes_ant`: si vendió exactamente lo mismo que el mes anterior
- `racha_crece` / `racha_cae`: meses consecutivos subiendo / bajando
- `pct_ceros_hist`: proporción de meses en cero en toda la historia

La racha de ceros ataca productos en declive/intermitentes, donde el modelo sobreestima.

In [ ]:
df = df.sort(['agrupa_id', 'periodo'])

# 1) Meses activo: cantidad de meses transcurridos desde la primera venta real (tn>0)
# Se cuenta la posición dentro del grupo (la serie ya arranca en el nacimiento por z301)
df = df.with_columns(
    pl.int_range(1, pl.len() + 1).over('agrupa_id').cast(pl.Int32).alias('meses_activo')
)

# 2) ¿Vendió exactamente lo mismo que el mes anterior?
df = df.with_columns(
    (pl.col('tn') == pl.col('tn').shift(1).over('agrupa_id'))
    .cast(pl.Int8).fill_null(0).alias('compro_igual_mes_ant')
)

# 3) Racha de ceros: meses consecutivos en cero hasta el actual
# Se usa un id de bloque que cambia cada vez que hay una venta; dentro del bloque de ceros se cuenta.
es_cero = (pl.col('tn') == 0).cast(pl.Int32)
df = df.with_columns(es_cero.alias('_es_cero'))
df = df.with_columns(
    # acumulado de no-ceros define el bloque; el cumsum de _es_cero dentro del bloque es la racha
    (1 - pl.col('_es_cero')).cum_sum().over('agrupa_id').alias('_bloque')
)
df = df.with_columns(
    pl.col('_es_cero').cum_sum().over(['agrupa_id', '_bloque']).cast(pl.Int32).alias('racha_ceros')
)

# 4) Rachas de tendencia: meses consecutivos subiendo o bajando
sube = (pl.col('tn') > pl.col('tn').shift(1)).over('agrupa_id')
baja = (pl.col('tn') < pl.col('tn').shift(1)).over('agrupa_id')
df = df.with_columns([
    sube.cast(pl.Int8).fill_null(0).alias('_sube'),
    baja.cast(pl.Int8).fill_null(0).alias('_baja'),
])
# racha de crecimiento: cumsum que se reinicia cuando deja de subir
df = df.with_columns([
    (1 - pl.col('_sube')).cum_sum().over('agrupa_id').alias('_blq_sube'),
    (1 - pl.col('_baja')).cum_sum().over('agrupa_id').alias('_blq_baja'),
])
df = df.with_columns([
    pl.col('_sube').cum_sum().over(['agrupa_id', '_blq_sube']).cast(pl.Int32).alias('racha_crece'),
    pl.col('_baja').cum_sum().over(['agrupa_id', '_blq_baja']).cast(pl.Int32).alias('racha_cae'),
])

# 5) Proporción histórica de ceros hasta el período actual (intermitencia acumulada)
df = df.with_columns(
    (pl.col('_es_cero').cum_sum().over('agrupa_id') /
     pl.int_range(1, pl.len() + 1).over('agrupa_id'))
    .cast(pl.Float32).alias('pct_ceros_hist')
)

# Limpiar columnas auxiliares
df = df.drop(['_es_cero', '_bloque', '_sube', '_baja', '_blq_sube', '_blq_baja'])

print('Features de comportamiento calculadas.')
cols_comp = ['meses_activo', 'racha_ceros', 'compro_igual_mes_ant',
             'racha_crece', 'racha_cae', 'pct_ceros_hist']
print(df.filter(pl.col('product_id') == 20001).select(['periodo', 'tn'] + cols_comp).head(8))

## 6. Escalado: media y desvío rolling

Calculado hacia atrás (con `shift(1)` para no incluir el período actual → sin leakage).
`media_rolling` y `std_rolling` quedan como features.
`tn_scaled = tn / media_rolling` (feature adicional, no reemplaza tn).

In [ ]:
ventana = PARAM['ventana_rolling']
w = len(df) if ventana is None else ventana

df = df.with_columns([
    pl.col('tn').shift(1).rolling_mean(window_size=w, min_periods=1)
      .over('agrupa_id').alias('media_rolling'),
    pl.col('tn').shift(1).rolling_std(window_size=w, min_periods=2)
      .over('agrupa_id').alias('std_rolling'),
])

df = df.with_columns(
    pl.when(pl.col('media_rolling') > 0)
      .then(pl.col('tn') / pl.col('media_rolling'))
      .otherwise(pl.lit(0.0))
      .alias('tn_scaled')
)

print('Escalado calculado.')
df.filter(pl.col('product_id') == 20001).select(
    ['periodo', 'tn', 'media_rolling', 'std_rolling', 'tn_scaled']
).head(8)

## 7. Features de calendario

In [ ]:
df = df.with_columns([
    (pl.col('periodo') % 100).cast(pl.Int8).alias('mes'),
    (pl.col('periodo') // 100).cast(pl.Int16).alias('anio'),
])
print('Features de calendario: mes, anio')

## 8. Targets: nivel y delta (ambos como columnas)

Se calculan **los dos** targets siempre. z303 elige cuál usar con la palanca `tipo_target`.
- `target_nivel` = tn en t+2
- `target_delta` = tn(t+2) - tn(t)

In [ ]:
h = PARAM['horizonte']

df = df.with_columns(
    pl.col('tn').shift(-h).over('agrupa_id').alias('tn_t2')
)

# Ambos targets como columnas (la elección se hace en z303)
df = df.with_columns([
    pl.col('tn_t2').alias('target_nivel'),
    (pl.col('tn_t2') - pl.col('tn')).alias('target_delta'),
])

# Train = filas con target conocido; Inferencia = últimas h filas por serie
df_train = df.filter(pl.col('tn_t2').is_not_null())
df_infer = df.filter(pl.col('tn_t2').is_null())

print(f'Train: {df_train.height:,} filas')
print(f'Inferencia: {df_infer.height:,} filas')
print('Targets disponibles: target_nivel, target_delta')

## 9. Limpieza

`lag_0` nunca es null (= tn). No removemos filas por lags largos nulos: LGBM los maneja nativamente y removerlas perdería productos con historia corta.

In [ ]:
# lag_0 = tn, nunca null. No removemos nada por lags largos.
# Solo verificamos.
for k in [0, min([l for l in PARAM['lags'] if l > 0])]:
    nulos = df_train.filter(pl.col(f'lag_{k}').is_null()).height
    print(f'  lag_{k}: {nulos} nulls en train')
print(f'Train final: {df_train.height:,} filas')

## 10. Guardar outputs

In [ ]:
df_train.write_parquet(PARAM['path_output'])
print(f'✅ Train guardado: {PARAM["path_output"]}')
print(f'   Filas: {df_train.height:,}  |  Columnas: {len(df_train.columns)}')
print('   Columnas:', df_train.columns)

path_infer = PARAM['path_output'].replace('z302_features', 'z302_inferencia')
df_infer.write_parquet(path_infer)
print(f'✅ Inferencia guardada: {path_infer}')
print(f'   Filas: {df_infer.height:,}')

## 11. Verificación

In [ ]:
#df_v = pl.read_parquet(PARAM['path_output'])
#print('Schema:')
#print(df_v.schema)
#print('\nDistribución del target:')
#print(df_v['target'].describe())